# The patent application continuation table (TLS216_APPLN_CONTN)

Welcome to the application continuation table, designated as TLS216_APPLN_CONTN.

In a similar way to the TLS204_APPLN_PRIOR table, which establishes the priority links between applications, the links between parent and child applications for various types of relations, such as continuation (in part), divisional applications and internal priorities, are defined via the TLS216_APPLN_CONTN table.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the as models
from epo.tipdata.patstat.database.models import TLS216_APPLN_CONTN

## APPLN_ID (Primary Key)

APPLN_ID refers to the continuation application. As for tables TLS204 and TLS205, we join tables TLS216 and TLS201 via two different attributes: `parent_appln_id` for table TLS216 and `appln_id` for table TLS201. In this way we can retrieve in table TLS201 previous applications to which `appln_id`s in table TLS216 are linked via the `parent_appln_id` attribute.

In [2]:
# Import table TLS201_APPLN 
from epo.tipdata.patstat.database.models import TLS201_APPLN

contn_id = db.query(
    TLS201_APPLN.inpadoc_family_id,
    TLS216_APPLN_CONTN.parent_appln_id,
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_filing_date,
    TLS201_APPLN.earliest_filing_date
).join(
    TLS201_APPLN, TLS216_APPLN_CONTN.parent_appln_id == TLS201_APPLN.appln_id  # Join table TLS201 and TLS216 
).limit(10000)

contn_id_df = patstat.df(contn_id)
contn_id_df

,inpadoc_family_id,parent_appln_id,appln_id,appln_filing_date,earliest_filing_date
0,274306,46552385,46552385,2008-03-28,2004-09-10
1,1814893,906586927,906586927,1994-10-14,1994-10-14
2,485624198,485624198,485624198,2016-04-29,2016-04-29
3,413389774,413389774,413389774,2013-02-28,2012-06-14
4,37089629,54429450,54429450,2001-10-22,2000-11-14
...,...,...,...,...,...
9995,490848620,502385335,502385335,2017-09-26,2015-06-12
9996,46822289,52288817,52288817,2003-07-18,2003-07-18
9997,266166,50615251,50615251,2006-04-13,2005-09-30
9998,422425773,456785163,456785163,2016-01-25,2014-09-04


## PARENT_APPLN_ID

PARENT_APPLN_ID refers to the application of which the APPLN_ID is a continuation. The two foreign keys (applications) should be different ones, i.e., there is no "self-continuation".

Also, for continuations, we can see that the applications referring to the same prior application belong to the same INPADOC family but possibly to different DOCDB families.

In [3]:
doc_inpa = db.query(
    TLS201_APPLN.appln_id,
    TLS216_APPLN_CONTN.parent_appln_id,
    TLS201_APPLN.inpadoc_family_id,
    TLS201_APPLN.docdb_family_id
).join(
    TLS201_APPLN, TLS216_APPLN_CONTN.appln_id == TLS201_APPLN.appln_id
).filter(
    TLS201_APPLN.inpadoc_family_id == 98645
).order_by(
    TLS216_APPLN_CONTN.parent_appln_id
)

doc_inpa_df = patstat.df(doc_inpa)
doc_inpa_df

,appln_id,parent_appln_id,inpadoc_family_id,docdb_family_id
0,505767121,46108868,98645,37908869
1,470852042,46108868,98645,37908869
2,410457093,46108868,98645,37908869
3,470852042,410457093,98645,37908869
4,505767121,410457093,98645,37908869
...,...,...,...,...
921,566501132,909308544,98645,80114271
922,498657819,909308544,98645,63168063
923,594864518,909308544,98645,80114271
924,489924383,909308544,98645,61159498


## CONTN_TYPE

The type of continuation describing what relation the later application has to the earlier application. 

Note that before 1991, the EPO did not record the so-called "linkage type" of priority numbers, which means that it was not recorded which kind of relation a given priority number has (Paris Union priority, continuation, division, etc.). Data in this element prior to 1991 is thus not reliable.

If the continuation type is not known, then `contn_type` is filled with three spaces. Let's take a look at the continuation applications with a continuation type assigned.

In [4]:
contn_type = db.query(
    TLS216_APPLN_CONTN.parent_appln_id,
    TLS216_APPLN_CONTN.appln_id,
    TLS216_APPLN_CONTN.contn_type
).filter(
    TLS216_APPLN_CONTN.contn_type != '   '  # Filter selecting those applications with contn_type attribute that is not blank (three spaces)
).order_by(
    TLS216_APPLN_CONTN.contn_type
).limit(100000)

contn_type_df = patstat.df(contn_type)
contn_type_df

,parent_appln_id,appln_id,contn_type
0,909212333,6378841,ADD
1,421391611,516421966,ADD
2,901323163,6425935,ADD
3,909109783,6331139,ADD
4,901243870,6305560,ADD
...,...,...,...
99995,909342459,298711054,CIP
99996,55320237,52121474,CIP
99997,49162713,340289954,CIP
99998,909193864,481180480,CIP


In the part of the table that is displayed, we can see two types of continuation: addition (ADD) and continuation in part (CIP).

Changing the filter, we can see that there are many continuation applications with unknown type.

In [5]:
# Import func to use count
from sqlalchemy import func

unkwn_contn_type = db.query(
    func.count(TLS216_APPLN_CONTN.contn_type).label('tot_num_of_unknown_types')
).filter(
    TLS216_APPLN_CONTN.contn_type == '   '
)

unkwn_contn_type_df = patstat.df(unkwn_contn_type)
unkwn_contn_type_df = unkwn_contn_type_df['tot_num_of_unknown_types'].item()
print("There are "+str(unkwn_contn_type_df)+" continuation applications with unknown continuation type")

There are 165748 continuation applications with unknown continuation type
